In [106]:
from euriai.langchain import create_chat_model

chat_model = create_chat_model(
    api_key="euri-4a6b60fdc171add4998299fbf26470f03f2daed7cff2f87a80406e4b07428f21",
    model="gpt-4.1-nano",
    temperature=0.7
)

response = chat_model.invoke("What is artificial intelligence?")
print(response.content)


Artificial intelligence (AI) refers to the simulation of human intelligence processes by computer systems and machines. This includes tasks such as learning from data (machine learning), understanding natural language, recognizing patterns, solving problems, and making decisions. AI can be categorized into narrow AI, which is designed for specific tasks (like virtual assistants or recommendation systems), and general AI, which would have the ability to perform any intellectual task a human can do—though this level of AI remains theoretical at present. Overall, AI aims to create systems that can perform tasks that typically require human intelligence, enhancing automation, efficiency, and decision-making across various fields.


In [2]:
import pydantic
print(pydantic.__version__)

2.11.7


In [3]:
pip install --upgrade euriai


Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast,re,math,os,sys
os.environ["euri-api-key"] ="euri-54b13eac3fbedb9873948ebd6e5f311f670f336cdcedb2dedf3ca864bb38d83a"
from euriai.langchain import create_chat_model
from langchain.agents import create_react_agent,AgentExecutor
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda,RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory ,InMemoryChatMessageHistory as ChatMessageHistory
from langchain_core.tools import BaseTool,tool
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    CombinedMemory,
    ReadOnlySharedMemory,
    ConversationEntityMemory

)


In [ ]:
pip install -U langchain langchain-core langchain-community langchain-openai

In [107]:


from euriai.langchain import create_chat_model

llm_default = create_chat_model(
    api_key="euri-4a6b60fdc171add4998299fbf26470f03f2daed7cff2f87a80406e4b07428f21",
    model="gpt-4.1-nano",
    temperature=0.7
)


In [108]:
@tool("calculator",return_direct=True)
def calculator(expression: str) -> str:
    """Evaluate a numeric math expression . supprts + , -,*,/,**,parentheses and all kind of mathamatical functions"""
    allowed_nodes =(
        ast.Expression,
        ast.UnaryOp,
        ast.unaryop,
        ast.BinOp,
        ast.operator,
        ast.Num,
        ast.Load,
        ast.pow,
        ast.FunctionDef,
        ast.Module,
        ast.Expr,
        ast.Call,
        ast.Name,
        ast.arguments,
        ast.args,
        ast.Constant
    )
    
    allowed_names = {k:v for k,v in vars(math).items() if not k.startswith("_")}
    allowed_names.update({"abs": abs, "round": round,"min": min,"max": max})
    node = ast.parse(expression, mode="eval")
    
    
    for n in ast.walk(node):
        if not isinstance(n, allowed_nodes):
            raise ValueError(f"Expression contains invalid node {type(n)}")
        if isinstance(n, ast.Name) and n.id not in allowed_names:
            raise ValueError(f"Expression contains invalid name {n.id}")
    code = compile(node, "<string>", "eval")
    return str(eval(code, {"__builtins__": {}}, allowed_names))

In [109]:
tools_math = [calculator]

In [110]:
react_prompt_math = ChatPromptTemplate.from_messages([(
    "system","you are a precise math assistant , you can use thse tools:\n{tools}\n"
    "when you are going to use this tools follw these instruction exactly the same format:\n"
    "Questions :......\nthought ...\nAction by using one of the tools access that you have [{tool_names}]\n "
    "alwasy finish with final give me a numeric answer to the question"),
    ("human","questions: {input}\n{agent_scratchpad}")
])

In [111]:
agent_math = create_react_agent(
    llm=llm_default,
    prompt=react_prompt_math,
    tools=tools_math
)

In [112]:
memory_math = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="input",
    output_key="output",
)

In [113]:
math_exector = AgentExecutor(
    agent=agent_math,
    tools=tools_math,
    memory=memory_math,
    verbose=True,
    handle_parsing_errors=(
        "you did not follow the instruction that i have given you to use the tools"
    ),
)

In [114]:
@tool("kb_search", return_direct=True)
def kb_search(query: str) -> str:
    """a mock function to search from a knowledge base"""
    knowledge_base = {
        "what is the capital of france": "The capital of France is Paris.",
        "who is the president of the united states": "The president of the United States is Joe Biden.",
        "what is the largest mammal": "The largest mammal is the blue whale.",
    }
    return knowledge_base.get(query.lower(), "I don't know the answer to that question.")

In [115]:
tools_kb = [kb_search]

In [116]:
react_prompt_kb = ChatPromptTemplate.from_messages([
    ("system","you are a helpful assistant that can answer question based on your knowledge base and you can use the following tools:\n{tools}\n"
    "when you are going to use this tools follw these instruction exactly the same format:\n"
    "Questions :......\nthought ...\nAction by using one of the tools access that you have [{tool_names}]\n "
    "alwasy finish with final give me a numeric answer to the question"),
    ("human","questions: {input}\n{agent_scratchpad}")
])

In [117]:
agent_kb = create_react_agent(
    llm=llm_default,
    prompt=react_prompt_kb,
    tools=tools_kb
)

In [118]:
mem_kb = ConversationBufferWindowMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="input",
    output_key="output",
    k=5
)

In [119]:
kb_executor = AgentExecutor(
    agent=agent_kb,
    tools=tools_kb,
    memory=mem_kb,
    verbose=True,
    handle_parsing_errors=(
        "you did not follow the instruction that i have given you to use the kb  tools"
    ),
)

In [120]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a router. Read the user message and output exactly one token:\n"
     "- MATH: if it asks for calculations.\n"
     "- KB: otherwise (tech education, RAG, agents, general know-how).\n"
     "Output only MATH or KB."),
    ("human", "{input}")
])

In [121]:
router_chain = router_prompt|llm_default|StrOutputParser()

In [129]:
def _dispatcher(inputs:dict):
    """it will take new inputs from human and it will also attach hitory from memory """
    user_msg = inputs["input"]
    choice = router_chain.invoke({"input": user_msg}).strip().upper()
    print(f"router choice : {choice}")
    if choice == "MATH" or choice == "Math" or choice == "math":
        return math_exector.invoke({"input": user_msg})
    elif choice == "KB" or choice == "Kb" or choice == "kb":
        return kb_executor.invoke({"input": user_msg})
    else:
        return "I can only answer math and kb related questions"

In [130]:
dispatcher = RunnableLambda(_dispatcher)

In [131]:
_sessions = {}

In [132]:
def _get_history(session_id:str):
    if session_id not in _sessions:
        _sessions[session_id] = ChatMessageHistory()
    return _sessions[session_id]

In [133]:
orchestrator = RunnableWithMessageHistory(
    runnable=dispatcher,
    get_session_history=_get_history,
    input_key="input",
    history_messages_key="history"
)

In [134]:
cfg = {"configurable":{"session_id":"user1"}}

In [135]:
print(orchestrator.invoke({"input":"what is the capital of france?"},config=cfg))

router choice : KB


> Entering new AgentExecutor chain...
Could not parse LLM output: `Questions: what is the capital of france?  
thought ...  
Action by using one of the tools access that you have [kb_search]  
query: capital of France  
Final answer: 2`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE you did not follow the instruction that i have given you to use the kb  toolsCould not parse LLM output: `Questions: what is the capital of France?  
thought ...  
Action by using one of the tools access that you have [kb_search]  
query: capital of France  
Final answer: 1`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE you did not follow the instruction that i have given you to use the kb  toolsCould not parse LLM output: `Questions: what is the capital of France?  
thought ...  
Action by using one of the tools access that you have [kb_search]  
query: capital of 

In [136]:
print(orchestrator.invoke({"input":"what is sin(90)"},config=cfg))

router choice : MATH


> Entering new AgentExecutor chain...
Could not parse LLM output: `thought ...  
To find sin(90 degrees), I need to evaluate the sine of 90 degrees. Since the calculator works with radians by default, I will convert 90 degrees to radians.  
90 degrees = π/2 radians.  
Now, I will calculate sin(π/2).  

Action by using one of the tools: [calculator]  
sin(π/2)

The sine of π/2 radians is 1.  

Final answer: 1`
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE you did not follow the instruction that i have given you to use the toolsCould not parse LLM output: `thought ...  
To find sin(90 degrees), I need to evaluate the sine of 90 degrees. Since the calculator works with radians by default, I will convert 90 degrees to radians.  
90 degrees = π/2 radians.  
Now, I will calculate sin(π/2).  

Action by using one of the tools: [calculator]  
sin(π/2)  

The sine of π/2 radians is 1.  

Final answer: 1`
For tr